# 02 — Pipeline Spark : jointures, indicateurs et score d'impact

**SciPulse Insights — Projet #4** · Yahya Qabil

Ce notebook documente la couche **Gold** produite par Spark (`src/scipulse/spark/`) :

1. Choix et justification de la **source de citations** ;
2. **Jointure à trois sources** et taux de correspondance (*match coverage*) ;
3. **Stratégie de jointure** : broadcast vs sort-merge, lecture des plans d'exécution (`.explain()`) ;
4. **Indicateurs temporels** avec des *window functions* ;
5. **TF-IDF distribué** (Spark MLlib) sur les résumés ;
6. **Score d'impact composite** : formule, justification des pondérations, **analyse de sensibilité**.

> **Exécution** : ce notebook tourne **dans le conteneur Airflow**, qui contient Java et PySpark et accède au cluster
> Spark et à MinIO :
> ```
> docker compose exec airflow-scheduler jupyter nbconvert --to notebook --execute --inplace \
>     /opt/airflow/notebooks/02_spark_pipeline.ipynb --ExecutePreprocessor.timeout=1800
> ```

## 1. Choix de la source de citations : OpenAlex

| Critère | **OpenAlex** (retenu) | Semantic Scholar |
|---|---|---|
| Accès | API REST ouverte, sans clé (pool « polite » avec `mailto`) | Clé optionnelle ; sans clé, quotas partagés et erreurs 429 fréquentes |
| Requête par lot | Filtre `doi:a\|b\|c` (50 DOI par requête) via le DOI arXiv `10.48550/arxiv.<id>` | `POST /paper/batch` (500 ids) |
| Vélocité de citation | `counts_by_year` (citations par année) | Total seulement |
| Graphe de citations | `referenced_works` (utile pour l'option B : PageRank) | Endpoint séparé, coûteux |
| Licence | CC0 (données ouvertes) | Licence propre |

**Difficulté rencontrée** : le champ `doi` d'OpenAlex pointe vers la **version publiée**, pas vers le DOI arXiv. La clé de
jointure est donc retrouvée dans `locations` (`arxiv.org/abs/<id>`) à l'ingestion (`ingestion/citations.py`).

In [ ]:
import os, sys, time
from pathlib import Path
sys.path.insert(0, "/opt/airflow/src" if Path("/opt/airflow/src").exists() else str(Path("..").resolve() / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession, functions as F, Window

from scipulse.config import buckets
from scipulse.spark.join import aggregate_hn, match_coverage
from scipulse.spark.indicators import yearly_growth_by_category, add_citation_velocity
from scipulse.spark.tfidf import fit_tfidf, add_top_terms, build_tfidf_pipeline
from scipulse.spark.impact_score import compute_impact

sns.set_theme(style="whitegrid", palette="deep")
FIG = Path("../docs/figures"); FIG.mkdir(parents=True, exist_ok=True)

LOCAL_SILVER = os.getenv("SCIPULSE_SILVER_DIR")   # uniquement pour tester le notebook hors Docker
if LOCAL_SILVER:
    spark = SparkSession.builder.master("local[2]").appName("nb02").getOrCreate()
    path = lambda b: f"{LOCAL_SILVER}/{b}"
else:
    from scipulse.common.spark import get_spark, s3a
    spark = get_spark("scipulse-notebook-02")
    path = s3a
spark.sparkContext.setLogLevel("ERROR")
CURRENT_YEAR = pd.Timestamp.now().year
print("Spark", spark.version, "—", spark.sparkContext.master)

## 2. Chargement des trois couches Silver

In [ ]:
arxiv = spark.read.parquet(path(buckets.ARXIV_CLEAN))
hn = spark.read.parquet(path(buckets.HN_CLEAN))
cit = spark.read.parquet(path(buckets.CITATIONS_CLEAN))

pd.DataFrame({
    "lignes": [arxiv.count(), hn.count(), cit.count()],
    "colonnes": [len(arxiv.columns), len(hn.columns), len(cit.columns)],
    "partitions Spark": [arxiv.rdd.getNumPartitions(), hn.rdd.getNumPartitions(), cit.rdd.getNumPartitions()],
}, index=["arxiv-clean", "hn-clean", "citations-clean"])

## 3. Jointure à trois sources

Un article peut être posté plusieurs fois sur HN : on **agrège d'abord HN par `arxiv_id`** (somme des scores et des
commentaires, nombre de posts), puis on fait deux **LEFT JOIN** depuis ArXiv. On conserve ainsi tous les articles, même
ceux sans citation ni mention HN.

```
arxiv  LEFT JOIN  citations  ON arxiv_id     (deux tables de taille comparable)
       LEFT JOIN  hn_agg     ON arxiv_id     (petite table agrégée)
```

In [ ]:
hn_agg = aggregate_hn(hn).cache()
cit_sel = cit.select("arxiv_id", "cited_by_count", "counts_by_year")
print(f"HN agrégé : {hn_agg.count():,} articles distincts")

def build_join(strategy):
    left = arxiv.select("arxiv_id", "primary_category", "year", "submitted_date", "abstract")
    h = F.broadcast(hn_agg) if strategy == "broadcast" else hn_agg
    return (left.join(cit_sel, "arxiv_id", "left").join(h, "arxiv_id", "left")
            .fillna({"cited_by_count": 0, "hn_score": 0, "hn_comments": 0, "hn_posts": 0}))

from contextlib import contextmanager
@contextmanager
def sort_merge_only():
    # Désactive le broadcast automatique ET l'exécution adaptative (AQE), qui convertirait sinon
    # le sort-merge en broadcast à l'exécution dès que la table est petite.
    old = {k: spark.conf.get(k) for k in ("spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.enabled")}
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    try:
        yield
    finally:
        for k, v in old.items():
            spark.conf.set(k, v)

gold = build_join("broadcast").cache()
coverage = match_coverage(gold)
pd.Series(coverage).to_frame("valeur")

**Lecture de la couverture** : presque tous les articles cités sur HN sont présents (ils sont **toujours conservés** à
l'ingestion Bronze). En revanche, la part d'articles avec **au moins une citation** reste faible : le corpus est
majoritairement **récent** (2024+) et la couverture OpenAlex est partielle (environ 45 % des articles demandés ont été
retrouvés). Dans ce corpus, « 0 citation » signifie donc souvent « pas encore de données ».

## 4. Stratégie de jointure : broadcast vs sort-merge

- **Broadcast hash join** : la petite table est copiée sur chaque executor et la grande n'est **jamais redistribuée**
  (pas de *shuffle*). C'est idéal quand une table tient en mémoire, ici HN agrégé (quelques dizaines de milliers de lignes).
- **Sort-merge join** : les deux tables sont **redistribuées** par clé (`Exchange hashpartitioning`), **triées**, puis
  fusionnées. C'est robuste pour deux grandes tables, mais coûteux en shuffle.

On compare les plans physiques et les temps d'exécution.

In [ ]:
def physical_joins(df):
    # Extrait les opérateurs de jointure et d'échange du plan physique
    plan = df._jdf.queryExecution().executedPlan().toString()
    keep = ("Join", "Exchange", "BroadcastExchange", "Sort [")
    return "\n".join(l for l in plan.splitlines() if any(k in l for k in keep))

# On vide le cache : sinon Spark réutilise le plan de `gold` (déjà calculé en broadcast) au lieu de replanifier
spark.catalog.clearCache()
print("=== broadcast (HN agrégé diffusé) ===")
print(physical_joins(build_join("broadcast")), "\n")
with sort_merge_only():
    print("=== sort-merge (broadcast désactivé) ===")
    print(physical_joins(build_join("sort-merge")))

In [ ]:
spark.catalog.clearCache(); hn_agg.cache().count()
timings = {}
def run(strategy):
    t0 = time.perf_counter()
    build_join(strategy).agg(F.sum("hn_score"), F.sum("cited_by_count")).collect()
    return round(time.perf_counter() - t0, 2)

for _ in range(2):                       # 2 passages : le 1er chauffe la JVM, on garde le 2e
    timings["broadcast"] = run("broadcast")
    with sort_merge_only():
        timings["sort-merge"] = run("sort-merge")
gold = build_join("broadcast").cache()      # on remet en cache la version retenue pour la suite
pd.Series(timings, name="secondes (2e passage)").to_frame()

**Conclusion** : dans le plan *broadcast*, les jointures sont des `BroadcastHashJoin` : les petites tables sont diffusées
(`BroadcastExchange`) et la grande table ArXiv n'est **pas redistribuée**. Le plan *sort-merge* ajoute un **shuffle**
(`Exchange hashpartitioning`) et un **tri** (`Sort`) **de chaque côté** de chaque jointure, d'où un temps d'exécution
nettement plus long.

**Stratégie retenue** (`spark/join.py`) :

- **broadcast explicite** pour HN agrégé : petite table quelle que soit la taille du corpus ;
- **choix automatique** pour les citations : sur notre échantillon (≈ 22 000 lignes), la table passe sous
  `spark.sql.autoBroadcastJoinThreshold` (10 Mo) et Spark la diffuse aussi. Sur le corpus complet (citations pour
  2,7 millions d'articles), elle dépasserait le seuil et Spark basculerait en **sort-merge**, la stratégie adaptée à deux
  grandes tables.

> Remarque : avec l'exécution adaptative (**AQE**, activée par défaut en Spark 3.5), Spark peut convertir un sort-merge en
> broadcast **à l'exécution** s'il constate qu'une table est petite. C'est pourquoi la démonstration du sort-merge désactive
> à la fois le seuil de broadcast et l'AQE.

## 5. Indicateurs temporels (window functions)

In [ ]:
monthly = (arxiv.withColumn("month", F.date_trunc("month", "submitted_date"))
           .groupBy("primary_category", "month").count())
w = Window.partitionBy("primary_category").orderBy("month")
growth = (monthly.withColumn("prev", F.lag("count").over(w))
          .withColumn("mom_growth", (F.col("count") - F.col("prev")) / F.col("prev")))

top_cats = [r.primary_category for r in arxiv.groupBy("primary_category").count()
            .orderBy(F.desc("count")).limit(5).collect()]
g = (growth.where(F.col("primary_category").isin(top_cats) & (F.col("month") >= "2024-01-01")
                  & (F.col("month") < "2025-03-01")).toPandas())
pivot = g.pivot(index="month", columns="primary_category", values="count").sort_index()
ax = pivot.plot(marker="o", figsize=(11, 4))
ax.set(title="Publications mensuelles — 5 principales catégories", xlabel="", ylabel="Articles")
plt.tight_layout(); plt.savefig(FIG / "02_croissance_categories.png", dpi=120); plt.show()

In [ ]:
# Croissance moyenne mensuelle par catégorie (catégories représentant ≥ 0,5 % du corpus)
min_articles = max(20, int(arxiv.count() * 0.005))
summary = (growth.where((F.col("month") >= "2024-02-01") & (F.col("month") < "2025-02-01"))
           .groupBy("primary_category")
           .agg(F.sum("count").alias("articles"), F.avg("mom_growth").alias("croissance_moy_mensuelle"))
           .where(F.col("articles") >= min_articles).orderBy(F.desc("croissance_moy_mensuelle")).toPandas())
summary["croissance_moy_mensuelle"] = (summary["croissance_moy_mensuelle"] * 100).round(1)
summary.head(10)

In [ ]:
# Vélocité de citation = citations / années depuis publication ; citations des 2 dernières années (counts_by_year)
vel = add_citation_velocity(gold, CURRENT_YEAR)
(vel.where("cited_by_count > 0").select("citation_velocity", "citations_last_2y")
    .summary("count", "mean", "50%", "90%", "max").toPandas().set_index("summary"))

`lag()` sur une fenêtre `partitionBy(categorie).orderBy(mois)` donne la croissance **mois par mois** de chaque catégorie
sans auto-jointure. La même technique est utilisée en production (`spark/indicators.py`) pour la croissance annuelle
et pour le rang d'un article dans sa catégorie (`row_number()`).

## 6. TF-IDF distribué (Spark MLlib)

In [ ]:
pipeline = build_tfidf_pipeline()
print(" → ".join(type(s).__name__ for s in pipeline.getStages()))
sample = gold.sample(fraction=min(1.0, 60_000 / max(gold.count(), 1)), seed=42)
model, tfidf_df = fit_tfidf(sample)
terms = add_top_terms(tfidf_df, model, k=5)
print(f"Vocabulaire : {len(model.stages[2].vocabulary):,} termes (échantillon de {sample.count():,} résumés)")

In [ ]:
# Termes les plus distinctifs par catégorie : fréquence des top-5 termes TF-IDF des articles de la catégorie
top_terms = (terms.where(F.col("primary_category").isin(top_cats))
             .select("primary_category", F.explode("top_terms").alias("term"))
             .groupBy("primary_category", "term").count()
             .withColumn("rk", F.row_number().over(Window.partitionBy("primary_category").orderBy(F.desc("count"))))
             .where("rk <= 8").toPandas())
top_terms.groupby("primary_category")["term"].apply(lambda s: ", ".join(s)).to_frame("termes distinctifs")

Pipeline : `RegexTokenizer → StopWordsRemover → CountVectorizer → IDF`. Le **CountVectorizer** (plutôt que
`HashingTF`) garde le vocabulaire, ce qui permet de **relire les termes** et de les exposer dans Kibana (`top_terms`).
`minDF=5` écarte les termes trop rares (fautes de frappe, identifiants), et le poids IDF fait baisser les termes génériques
du domaine (*model*, *method*…).

## 7. Score d'impact composite

$$\text{impact} = 100 \times \big(\,0{,}5 \cdot \text{cit\_norm} + 0{,}3 \cdot \text{hn\_norm} + 0{,}2 \cdot \text{recency}\,\big)$$

| Composante | Calcul | Justification |
|---|---|---|
| `cit_norm` | `percent_rank` de `log1p(citations)` **dans la catégorie** | Les volumes de citations varient énormément selon les domaines ; le log écrase les valeurs extrêmes |
| `hn_norm` | `percent_rank` de `log1p(score + commentaires)` parmi les articles présents sur HN, 0 sinon | Signal de popularité tech, en log comme demandé par le brief |
| `recency` | $e^{-\ln 2 \cdot \text{âge} / 2}$ (demi-vie de 2 ans) | Ne pas pénaliser les articles trop récents pour avoir été cités |

**Pondérations** : le signal académique reste le principal (0,5), l'écho tech est un complément fort (0,3) et la récence
un correctif (0,2).

In [ ]:
scored = compute_impact(add_citation_velocity(gold, CURRENT_YEAR), CURRENT_YEAR).cache()
pdf = scored.select("arxiv_id", "primary_category", "year", "cited_by_count", "hn_score",
                    "cit_norm", "hn_norm", "recency", "impact_score", "is_hidden_gem", "is_hn_hype").toPandas()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, ["cit_norm", "hn_norm", "recency", "impact_score"]):
    pdf[col].plot.hist(bins=40, ax=ax); ax.set_title(col)
plt.tight_layout(); plt.savefig(FIG / "02_composantes_score.png", dpi=120); plt.show()
pd.Series({"pépites cachées": int(pdf.is_hidden_gem.sum()), "hype HN": int(pdf.is_hn_hype.sum())}).to_frame("articles")

### Analyse de sensibilité des pondérations

On recalcule le score avec d'autres jeux de poids et on compare au classement de référence :
- **recouvrement du top 100** (part des articles communs) ;
- **corrélation de Spearman** des rangs sur l'ensemble du corpus.

In [ ]:
WEIGHTS = {
    "référence (0.5/0.3/0.2)": (0.5, 0.3, 0.2),
    "académique (0.7/0.2/0.1)": (0.7, 0.2, 0.1),
    "tech (0.3/0.6/0.1)": (0.3, 0.6, 0.1),
    "sans récence (0.6/0.4/0)": (0.6, 0.4, 0.0),
}
for name, (wc, wh, wr) in WEIGHTS.items():
    pdf[name] = 100 * (wc * pdf.cit_norm + wh * pdf.hn_norm + wr * pdf.recency)

ref = list(WEIGHTS)[0]
top_ref = set(pdf.nlargest(100, ref).arxiv_id)
sens = pd.DataFrame({
    "recouvrement top 100": {n: len(top_ref & set(pdf.nlargest(100, n).arxiv_id)) / 100 for n in WEIGHTS},
    "Spearman vs référence": pdf[list(WEIGHTS)].corr(method="spearman")[ref].round(3),
    "année médiane du top 100": {n: int(pdf.nlargest(100, n).year.median()) for n in WEIGHTS},
    "citations médianes du top 100": {n: float(pdf.nlargest(100, n).cited_by_count.median()) for n in WEIGHTS},
})
sens

**Interprétation** (à compléter avec tes chiffres) :

- Une **corrélation de Spearman élevée** entre les variantes indique que le classement global est **robuste** : seules
  les premières places changent.
- Le top 100 de référence est **dominé par des articles récents et viraux sur HN** : dans un corpus où la plupart des
  articles ont 0 citation, quelques citations suffisent à atteindre un percentile élevé dans sa catégorie (distribution
  **concentrée sur zéro**). Le score mesure donc surtout « ce qui compte **maintenant** », ce qui convient à une
  plateforme de **veille**.
- La variante **sans récence** fait remonter les classiques très cités : c'est l'option à privilégier pour un classement
  d'**influence historique**.

## 8. Écriture Gold et orchestration

Le DataFrame final est écrit dans l'index Elasticsearch `arxiv-papers-enriched` via le connecteur
**elasticsearch-hadoop** (`spark/write_es.py`, `es.mapping.id = arxiv_id` donc ré-exécution idempotente). Le job
(`spark/gold_job.py`) est orchestré par le DAG `scipulse_pipeline` : HN Silver → citations Bronze → citations Silver → Spark
Gold.

## 9. Limites et pistes

| Limite | Effet | Piste |
|---|---|---|
| Couverture OpenAlex ≈ 45 % | « 0 citation » ≠ « non cité » (ex. DeepSeek-R1 dans la Hype HN) | Champ `has_citation_data`, compléter avec Semantic Scholar |
| Plafond de 200 000 articles | Corpus coupé début 2025 | Augmenter `ARXIV_MAX_RECORDS` (ressources) |
| Percentile sur distribution concentrée en 0 | Quelques citations suffisent à atteindre un percentile élevé | Lissage bayésien, ou percentile calculé sur les articles cités uniquement |
| Récence | Favorise les articles récents | Proposer deux scores : « veille » et « influence » |

In [ ]:
spark.catalog.clearCache()
spark.stop()